In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from tabulate import tabulate

Chargement des données

MOVIES

In [11]:
movies = pd.read_csv('movies.csv')

# Remplacement du genre (no genres listed) par NaN
movies['genres'] = movies['genres'].replace('(no genres listed)', np.nan)

# Séparation des genres en listes
movies = movies.join(
    movies['genres'].str.get_dummies(sep='|').add_prefix('genre_')
)

movies = movies.drop(columns=['genres'])

display(movies.head())

movies.info()

,movieId,title,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,...,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


<class 'pandas.DataFrame'>
RangeIndex: 27278 entries, 0 to 27277
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   movieId            27278 non-null  int64
 1   title              27278 non-null  str  
 2   genre_Action       27278 non-null  int64
 3   genre_Adventure    27278 non-null  int64
 4   genre_Animation    27278 non-null  int64
 5   genre_Children     27278 non-null  int64
 6   genre_Comedy       27278 non-null  int64
 7   genre_Crime        27278 non-null  int64
 8   genre_Documentary  27278 non-null  int64
 9   genre_Drama        27278 non-null  int64
 10  genre_Fantasy      27278 non-null  int64
 11  genre_Film-Noir    27278 non-null  int64
 12  genre_Horror       27278 non-null  int64
 13  genre_IMAX         27278 non-null  int64
 14  genre_Musical      27278 non-null  int64
 15  genre_Mystery      27278 non-null  int64
 16  genre_Romance      27278 non-null  int64
 17  genre_Sci-Fi       2727

RATINGS

In [12]:
ratings = pd.read_csv('ratings.csv')

ratings = ratings.drop(columns=['timestamp'])

display(ratings.head())
ratings.info(show_counts=True)

,userId,movieId,rating
0,1,2,3.5
1,1,29,3.5
2,1,32,3.5
3,1,47,3.5
4,1,50,3.5


<class 'pandas.DataFrame'>
RangeIndex: 20000263 entries, 0 to 20000262
Data columns (total 3 columns):
 #   Column   Non-Null Count     Dtype  
---  ------   --------------     -----  
 0   userId   20000263 non-null  int64  
 1   movieId  20000263 non-null  int64  
 2   rating   20000263 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 457.8 MB


LINKS

In [13]:
links = pd.read_csv('links.csv')

links = links.drop(columns=['tmdbId'])

display(links.head())
links.info()

,movieId,imdbId
0,1,114709
1,2,113497
2,3,113228
3,4,114885
4,5,113041


<class 'pandas.DataFrame'>
RangeIndex: 27278 entries, 0 to 27277
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  27278 non-null  int64
 1   imdbId   27278 non-null  int64
dtypes: int64(2)
memory usage: 426.3 KB


TAGS

In [16]:
tags = pd.read_csv('tags.csv')

tags = tags.drop(columns=['timestamp'])
tags = tags.dropna()

display(tags.head())
tags.info()

,userId,movieId,tag
0,18,4141,Mark Waters
1,65,208,dark hero
2,65,353,dark hero
3,65,521,noir thriller
4,65,592,dark hero


<class 'pandas.DataFrame'>
Index: 465548 entries, 0 to 465563
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   userId   465548 non-null  int64
 1   movieId  465548 non-null  int64
 2   tag      465548 non-null  str  
dtypes: int64(2), str(1)
memory usage: 14.2 MB


SCORE MOVIE : GENOME SCORES & GENOME TAGS

In [22]:
genome_scores = pd.read_csv('genome-scores.csv')
genome_tags = pd.read_csv('genome-tags.csv')

display(genome_scores.head())
genome_tags.head()

,movieId,tagId,relevance
0,1,1,0.02500
1,1,2,0.02500
2,1,3,0.05775
3,1,4,0.09675
4,1,5,0.14675


,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s


**********************************************************************

Filtrage basé sur le contenu : on utilise l'information qu'on connaît sur les intérêts des utilisateurs comme liaison pour les recommandations potentielles. Ces intérêts sont dans le dataframe tag

**********************************************************************

dans le dataframe tag nous avons plusieurs lignes pour un film, soit une ligne par tag. On va regrouper les tags en une seule ligne par film

In [26]:
tagsMovie = (
    tags
    .groupby('movieId')['tag']
    .apply(lambda x: ' '.join(x.astype(str)))
    .reset_index()
)
display(tagsMovie.head(2))
tagsMovie.shape

,movieId,tag
0,1,Watched computer animation Disney animated fea...
1,2,time travel adapted from:book board game child...


(19545, 2)

In [27]:
# ajout du titre du film
tagsMovie = tagsMovie.merge(movies[['movieId', 'title']], on='movieId', how='left')
display(tagsMovie.head(2))
tagsMovie.shape

,movieId,tag,title
0,1,Watched computer animation Disney animated fea...,Toy Story (1995)
1,2,time travel adapted from:book board game child...,Jumanji (1995)


(19545, 3)

In [32]:
# tokenization
tfidf = TfidfVectorizer(stop_words='english')
matrice_tfidf = tfidf.fit_transform(tagsMovie['tag'])
print(matrice_tfidf.shape)

# similarité des films
# On calcule la similarité cosinus
sim_cosinus = cosine_similarity(matrice_tfidf, matrice_tfidf)

# On calcule la similarité euclidienne
sim_euclidienne = 1 / (1 + euclidean_distances(matrice_tfidf))

(19545, 23863)


In [34]:
# Créer une série d'indices en utilisant la colonne 'title' comme index
indices = pd.Series(range(0,len(tagsMovie)), index=tagsMovie['title'])
display(indices.head(2))

# Fonction de recommandation basée sur la similarité cosinus
def recommander_films(titre, mat_sim=sim_cosinus, num_recommendations = 10):
    # Récupérer l'index du film correspondant au titre
    idx = indices[titre]
    
    # Obtenir les scores de similarité pour ce film
    scores_similarite = list(enumerate(mat_sim[idx]))
    
    # Trier les films par score de similarité
    scores_similarite = sorted(scores_similarite, key=lambda x: x[1], reverse=True)
    
    # Récupérer les scores des 10 films les plus similaires (en excluant le film lui-même)
    top_similair = scores_similarite[1:num_recommendations+1]
    res = [(indices.index[idx], score) for idx, score in top_similair]
    
    # Retourner les titres des films recommandés
    return tabulate(res, headers=["Titre", "Score de similarité"], tablefmt="pretty")

title
Toy Story (1995)    0
Jumanji (1995)      1
dtype: int64

In [54]:
# recherche d'un exemple de titre à tester
tagsMovie.loc[tagsMovie['title'].str.contains("bourne", case=False, na=False), 'title']

583                               Mrs. Winterbourne (1996)
4601                           Bourne Identity, The (2002)
6932                          Bourne Supremacy, The (2004)
7775                           Bourne Identity, The (1988)
10237                         Bourne Ultimatum, The (2007)
14593                            Bourne Legacy, The (2012)
16345                       God Bless Ozzy Osbourne (2011)
17002    Strange Case of Dr. Jekyll and Miss Osbourne, ...
19171                                     Melbourne (2014)
Name: title, dtype: str

In [57]:
#test
exple = 'Bourne Identity, The (2002)'
print(f"Recommandations pour {exple} similarité cosinus: \n",recommander_films(exple, sim_cosinus))
print(f"\n Recommandations pour {exple} similarité euclidienne: \n",recommander_films(exple, sim_euclidienne))

Recommandations pour Bourne Identity, The (2002) similarité cosinus: 
 +-------------------------------------+---------------------+
|                Titre                | Score de similarité |
+-------------------------------------+---------------------+
|    Bourne Supremacy, The (2004)     | 0.9200668130329155  |
|    Bourne Ultimatum, The (2007)     | 0.8170948300460554  |
|          Green Zone (2010)          | 0.4751289181965447  |
|         Double, The (2011)          | 0.4382597408981573  |
|         Stuck on You (2003)         | 0.41800349274664805 |
|      Monuments Men, The (2014)      | 0.3953662983664444  |
|     Bourne Identity, The (1988)     | 0.39226398929735434 |
| Tinker, Tailor, Soldier, Spy (1979) | 0.33795750465312285 |
|      Good Shepherd, The (2006)      | 0.3238446006720132  |
|   Talented Mr. Ripley, The (1999)   | 0.3176463504098267  |
+-------------------------------------+---------------------+

 Recommandations pour Bourne Identity, The (2002) similarité